### Peg-Solitaire Lösung suchen
Der Suchraum besteht allen legalen Spielpositionen.
Eine Spielposition speichern wir als 7x7 Matrix.
Dabei beachten wir nur Felder (x, y), wo mindestens eine Koordinate einen Wert in {2, 3, 4} hat.
Eine 0 oder 1 auf einem solchen Feld gibt an, es leer ist.  
```  0  1  2  3  4  5  6
0: [[?, ?, 1, 1, 1, ?, ?],
1:  [?, ?, 1, 1, 1, ?, ?],
2:  [1, 1, 1, 1, 1, 1, 1],
3:  [1, 1, 1, 0, 1, 1, 1],
4:  [1, 1, 1, 1, 1, 1, 1],
5:  [?, ?, 1, 1, 1, ?, ?],
6:  [?, ?, 1, 1, 1, ?, ?]]
```

Ob ein Koordinatentuple `pos= (x, y)` eine Spielposition ist, testen wir mit  
```python
all(0 <= i < 7 for i in pos) and any(2 <= i < 5 for i in pos)
```

Da wir hier mit Listen von Listen arbeiten, ist beim Kopieren der Spielposition
die Funktion `copy` aus unserem Modul `matrix_helpers` zu verwenden.
Und damit wir eine Spielposition als Schlüssel im `go_back` Dict verwenden können, verwandeln wir diese in
ein Tuple von Tuple mit
```python
tuple(tuple(row) for row in matrix)
```

Die Funktion `get_neighbors` liefert alle Spielpositionen, die von einer gegebenen Position aus erreichbar sind.
Genauer, sie liefert ein Tuple `(move, board)`. `move` ist der Zug, der zur Position `board` führt. 
Ein Zug ist dabei ein Triple 
> `(jpeg, peg, hole)` mit 3 Positionen (jumping peg, peg der entfernt wird, Loch).

Um rascher testen zu können, ob nur noch ein Stift übrig ist, speichern wir in der Liste `nodes_to_visit` Tuple  
der Form `(Anzahl Stifte, Spielposition)`.  
Im Dict go_back speichert wir Key-Value Paare der Form 
> `{as_tuple(new_board): (move, as_tuple(old_board))}`.

Der Zug `move` führt von `old_board` nach `new_board`. Die Funktion `get_path_home` passen wir entsprechend an.

In [ ]:
import matrix_helpers as M


def search_df(board, get_neighbors):
    n_pegs = 32
    nodes_to_visit = [(n_pegs, board)]
    go_back = {as_tuple(board): None}

    while nodes_to_visit:
        n_pegs, board = nodes_to_visit.pop()
        if n_pegs == 1 and M.get_item(board, (3, 3)) == 1:
            return board, go_back

        for move, new_board in get_neighbors(board):
            if (key := as_tuple(new_board)) in go_back:
                continue

            go_back[key] = (move, as_tuple(board))
            nodes_to_visit.append((n_pegs - 1, new_board))


def get_path_home(board, go_back):
    '''liefert  Liste mit Zügen'''
    node = go_back[as_tuple(board)]
    path = []

    while node:
        jpeg, peg, hole = node[0]
        path.append((jpeg, hole))
        node = go_back[node[1]]

    return path

In [ ]:
def new_board():
    board = M.make_matrix(7, default=1)
    M.set_item(board, (3, 3), 0)
    return board


def as_tuple(mat):
    return tuple(tuple(row) for row in mat)


def is_pos(pos):
    return all(0 <= i < 7 for i in pos) and any(2 <= i < 5 for i in pos)


def get_pegmoves(board, jpeg):
    x, y = jpeg
    moves = []
    for dx, dy in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        peg, hole = (x+dx, y+dy), (x+2*dx, y+2*dy)
        if is_pos(peg) and M.get_item(board, peg) == 1 and is_pos(hole) and M.get_item(board, hole) == 0:
            moves.append((jpeg, peg, hole))
    return moves


def get_neighbors(board):
    for jpeg, val in M.pos_and_values(board):
        if not is_pos(jpeg) or val == 0:
            continue

        moves = get_pegmoves(board, jpeg)
        for move in moves:
            jpeg, peg, hole = move
            new_board = M.copy(board)  # cast from tuple to list,
            M.set_item(new_board, jpeg, 0)
            M.set_item(new_board, peg, 0)
            M.set_item(new_board, hole, 1)
            yield move, new_board  #return a tuple!?

In [ ]:
board = new_board()
board[3][0] = 0
board[1][2] = 0
board

In [ ]:
jpeg = (1, 3)
get_pegmoves(board, jpeg)

In [ ]:
jpeg = (2, 3)
get_pegmoves(board, jpeg)

In [ ]:
ns = get_neighbors(board)
ns

In [ ]:
board

In [ ]:
move, board = next(ns)
print(move)
board

In [ ]:
board, go_back = search_df(new_board(), get_neighbors)
len(go_back)  # Anzahl besuchter Knoten

In [ ]:
board

In [ ]:
get_path_home(board, go_back)[::-1]